In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark import StorageLevel
from pyspark.sql.types import StringType,IntegerType
import time
spark = SparkSession.builder.appName("Spark-demo-app").getOrCreate()


In [2]:
df = spark.range(1, 1000000).withColumn("value", col("id") % 1000)


In [ ]:
df.take(32)

[Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9),
 Row(id=10, value=10),
 Row(id=11, value=11),
 Row(id=12, value=12)]

In [4]:
print("before partition:",df.rdd.getNumPartitions())

before partition: 2


In [5]:
df_repartition = df.repartition(20)

In [6]:
print("after partition:",df_repartition.rdd.getNumPartitions())

[Stage 1:>                                                          (0 + 2) / 2]

after partition: 20


In [7]:
df.write.mode("overwrite").csv("output/sample-data",header=True)

In [8]:
df_repartition.write.mode("overwrite").csv("output/sample-data1",header=True)

26/06/13 06:27:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [9]:
df_coalesced = df_repartition.coalesce(10)

In [10]:
print("after changing:",df_coalesced.rdd.getNumPartitions())

[Stage 6:=============================>                             (1 + 1) / 2]

after changing: 10


In [11]:
df_coalesced.write.mode("overwrite").csv("output/sample-data1",header=True)

In [12]:
df_optimized = df_repartition.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [13]:
df_optimized.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(20), REPARTITION_BY_NUM, [plan_id=191]
   +- Project [id#0L, (id#0L % 1000) AS value#2L]
      +- Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
         +- Range (1, 1000000, step=1, splits=2)




In [20]:
start_time = time.time()
count_uncached = df_optimized.count()
end_time = time.time()
print(f"1. optimized ecxection | count: {count_uncached} | time:{round(end_time-start_time,4)} seconds")

1. optimized ecxection | count: 499000 | time:0.8052 seconds


In [15]:
df_optimized.cache()

DataFrame[id: bigint, value: bigint]

In [19]:
start_time = time.time()
count_uncached = df_optimized.count()
end_time = time.time()
print(f"2. manual execution | count: {count_uncached} | time:{round(end_time-start_time,4)} seconds")

2. manual execution | count: 499000 | time:0.7381 seconds


In [17]:
df_optimized.unpersist()

DataFrame[id: bigint, value: bigint]

In [18]:
start_time = time.time()
count_uncached = df_optimized.count()
end_time = time.time()
print(f"3. after unpersist | count: {count_uncached} | time:{round(end_time-start_time,4)} seconds")

3. after unpersist | count: 499000 | time:0.5821 seconds


In [ ]:
data = [("Kunal",20),("Khushi",18),("Krishna",32),("Harshvardhan",10),("Dhruv",17)]
df1 = spark.createDataFrame(data,["Name","Age"])
df1.show()

+------------+---+
|        Name|Age|
+------------+---+
|       Kunal| 20|
|      Khushi| 21|
|     Krishna| 32|
|Harshvardhan| 10|
|       Dhruv| 17|
+------------+---+



In [27]:
def can_drive(Age):
    if(Age>18):
        return "Can Drive"
    elif(18>Age>12):
        return "Can Drive with learning"
    return "Cannot Drive"


In [30]:
can_drive_udf = udf(can_drive,StringType())

In [33]:
df_updated = df1.withColumn("Eligible",can_drive_udf(col("Age")))
df_updated.show()

+------------+---+--------------------+
|        Name|Age|            Eligible|
+------------+---+--------------------+
|       Kunal| 20|           Can Drive|
|      Khushi| 21|           Can Drive|
|     Krishna| 32|           Can Drive|
|Harshvardhan| 10|        Cannot Drive|
|       Dhruv| 17|Can Drive with le...|
+------------+---+--------------------+



In [34]:
spark.udf.register("sql_can_drive",can_drive,StringType())
df1.createOrReplaceTempView("people")

In [35]:
sql_df = spark.sql('''SELECT Name,Age,sql_can_drive(Age) AS Eligiblity
FROM people''')
sql_df.show()

+------------+---+--------------------+
|        Name|Age|          Eligiblity|
+------------+---+--------------------+
|       Kunal| 20|           Can Drive|
|      Khushi| 21|           Can Drive|
|     Krishna| 32|           Can Drive|
|Harshvardhan| 10|        Cannot Drive|
|       Dhruv| 17|Can Drive with le...|
+------------+---+--------------------+



In [36]:
spark.stop()